In [1]:
# !pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [2]:
# from langchain_.docments import Document

In [3]:
# Text Data
from langchain_community.document_loaders.text import TextLoader
loader = TextLoader("Data/python.txt", encoding="utf-8")

C:\Users\patil\AppData\Local\Temp\ipykernel_16972\210792745.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader


In [4]:
document = loader.load()
# document

In [5]:
# pdf Data
from langchain_community.document_loaders.pdf import PyPDFLoader

pdf_loader = PyPDFLoader("data/pdfs/research1.pdf")
document = pdf_loader.load()
# document

In [6]:
# pdf Data
from langchain_community.document_loaders.pdf import PyMuPDFLoader

pdf_loader = PyPDFLoader("data/pdfs/research1.pdf")
document = pdf_loader.load()
# document

# Building the Rag

## Ingestion Pipeline


In [7]:
# Data =>  Documents (langchain class)
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

In [8]:
def load_all_pdfs():
    folder_path = "Data/pdfs"
    num_docs = 0
    all_docs = []
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # Complete file path
            pdf_path = os.path.join(folder_path, filename)
            
            loader = PyPDFLoader(pdf_path)
            doc = loader.load()
            
            all_docs.extend(doc)
            num_docs += 1
    print("total pdfs:", num_docs)
    print("total pages:", len(all_docs))
    return all_docs

In [9]:
all_pdf_documents = load_all_pdfs()

total pdfs: 2
total pages: 32


In [10]:
type(all_pdf_documents[1])

langchain_core.documents.base.Document

## Chunking

In [11]:
# !pip install langchain_text_splitters

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size = 500, chunk_overlap = 50):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size =  chunk_size,
        chunk_overlap= chunk_overlap
    )
    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [13]:
chunks = split_docs(all_pdf_documents)

In [14]:
len(chunks)

321

## Embedding

In [15]:
from sentence_transformers import SentenceTransformer

In [16]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model_name = model_name
        print("Loading Model..",self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("Embedding Dimensions =", self.model.get_sentence_embedding_dimension)

        
    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar= True)
        print("embedding Shape:", embeddings.shape)
        return embeddings

In [17]:
embedding_manager = EmbeddingManager()

Loading Model.. all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding Dimensions = <bound method SentenceTransformer.get_sentence_embedding_dimension of SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)>


## Vector Store (Smaller Version of Vector Database)

In [18]:
import chromadb
import uuid

In [19]:
class VectorStoreManager:
    def __init__(self, persist_directory = "Data/vector_Store", collection_name = "pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None
        
        self._initialize_store()
        
    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok= True)

        # create a client
        self.client = chromadb.PersistentClient(path = self.persist_directory)

        # create the collection
        self.collection= self.client.get_or_create_collection(
            name = self.collection_name,
            metadata= {"description": "vector store collection for pdf embeddings in RAG"}
        )
        print("initialized the vector store with collection: ", self.collection_name)
        print("docs in collection", self.collection.count())
        
    # chunk => page_content, metadata
    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):  # because out cromadb expects that out length of documents, embeddings both should be same
            raise ValueError("Num of documents does not match no of embeddings")
        # In Vector Store what we store :-  1. Ids :- for every individual embedding we give diff id for uniqely identify.
        #                                   2. embeddings   3. documents 4. metadata 

        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []
                #  for individual chunk 
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # creating id
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            # metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            # document
            documents_content.append(doc.page_content)

            # embeddings
            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids = ids,
                metadatas= all_metadata,
                documents= documents_content,
                embeddings = embeddings_list
            )

        print("total document add in vector store", len(documents_content))
        print("docs in collection", self.collection.count())
            

In [20]:
vector_store = VectorStoreManager()

initialized the vector store with collection:  pdf_documents
docs in collection 642


In [21]:
texts = [doc.page_content for doc in chunks]

embedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, embedding)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

embedding Shape: (321, 384)
total document add in vector store 321
docs in collection 963


## Retrieval Pipeline

In [22]:
from sklearn.metrics.pairwise import cosine_similarity

In [23]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store
        
    def retrieve(self, query, top_k=5, score_threshold = 0.0):
        # convert query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semantic search perform
        results = self.vector_store.collection.query(
            query_embeddings = [query_embeddings.tolist()],
            n_results = top_k
        )
        # cosine similarity
        retrieved_docs = []
        # in such cases if does not found any similar results in symantic search for varifying that we apply this conditinal statement
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            
            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id" : doc_id,
                        "document" : document,
                        "distance" : distance,
                        "similarity_score" : similarity_score,
                        "rank" : i + 1
                    })
            print(f"retrieved {len(retrieved_docs)} documents")
        else:
            print("No document Found")
        return retrieved_docs   #b retrived document are our ""context""

In [24]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [25]:
rag_retriever.retrieve("What is encoder and decoder?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding Shape: (1, 384)
retrieved 5 documents


[{'id': 'doc_7b68962c-1bff-4897-90da-2e8fed687499',
  'document': 'respectively.\n3.1 Encoder and Decoder Stacks\nEncoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two\nsub-layers. The ﬁrst is a multi-head self-attention mechanism, and the second is a simple, position-\n2\npatil.durvesh8668@gmail.com',
  'distance': 0.7621868252754211,
  'similarity_score': 0.23781317472457886,
  'rank': 1},
 {'id': 'doc_ea3ad6af-6188-4d01-9e58-61311da2eb0d',
  'document': 'respectively.\n3.1 Encoder and Decoder Stacks\nEncoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two\nsub-layers. The ﬁrst is a multi-head self-attention mechanism, and the second is a simple, position-\n2\npatil.durvesh8668@gmail.com',
  'distance': 0.7621868252754211,
  'similarity_score': 0.23781317472457886,
  'rank': 2},
 {'id': 'doc_67da01b5-5961-49a5-b071-20fd2fa4149a',
  'document': 'respectively.\n3.1 Encoder and Decoder Stacks\nEncoder: The encoder is 

# Integrate with LLMs

## Open-AI Api

In [ ]:
API_KEY_OPENAI = ""

In [27]:
# !pip install langchain-openai

In [36]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    api_key=API_KEY_OPENAI,
    model="gpt-5.4",
    temperature=0.1,
    max_tokens=1024
)
print("ChatOpenAI ready")

ChatOpenAI ready


In [37]:
# Generate our retrieval-augmented output
def generate_output(query, rag_retriever, llm, top_k = 3):
    results = rag_retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use given context to generate the ans for the query
                    Context :{context}
                    Query : {query}
        """
    response = llm.invoke(prompt) # expecting a string as prompt
    return response.content

In [38]:
ans = generate_output("what is rag", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding Shape: (1, 384)
retrieved 3 documents


In [39]:
ans

'RAG stands for **Retrieval-Augmented Generation**.\n\nIt is a method that combines:\n\n- **Retrieval**: fetching relevant information from external sources such as documents, databases, or knowledge bases\n- **Generation**: using a language model to generate an answer based on the retrieved information\n\nFrom the given context, RAG is described as an evolving area with methods such as **naive RAG** and other advanced paradigms.\n\nIn simple terms, **RAG helps AI produce better, more factual answers by first looking up useful information and then generating a response from it**.'

In [40]:
ans = generate_output("what is encoder-decoder?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding Shape: (1, 384)
retrieved 3 documents


In [41]:
print(ans)

The encoder-decoder is an architecture where:

- The **encoder** reads and processes the input sequence into representations.
- The **decoder** uses those encoder outputs to generate the output sequence.

From the given context:
- The **decoder** has **6 identical layers**.
- Each decoder layer includes:
  1. a self-attention sub-layer,
  2. an encoder-decoder attention sub-layer that attends to the **output of the encoder stack**,
  3. another processing sub-layer.
- Both encoder and decoder use **residual connections** and **layer normalization**.
- The model output dimension is **d_model = 512**.

So, **encoder-decoder** refers to a model in which the encoder converts the input into internal representations, and the decoder attends to those representations to produce the final output.


In [42]:
response = llm.invoke("Hello")
print(response.content)

Hello! How can I help you today?


## GROQ

In [ ]:
GROQ_KEY_OPENAI =""

In [58]:
# !pip install -qU langchain-groq

In [59]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    api_key= GROQ_KEY_OPENAI,
    model="qwen/qwen3-32b",
    temperature=0.1,
    max_tokens=1024
)
print("ChatGroq ready")

ChatGroq ready


In [60]:
# Generate our retrieval-augmented output
def generate_output(query, rag_retriever, llm, top_k = 3):
    results = rag_retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use given context to generate the ans for the query
                    Context :{context}
                    Query : {query}
        """
    response = llm.invoke([prompt.format(context = context, query = query)]) # Groq expecting a list as prompt 
    return response.content

In [61]:
ans = generate_output("what is encoder-decoder?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding Shape: (1, 384)
retrieved 3 documents


In [62]:
print(ans)

<think>
Okay, the user is asking, "what is encoder-decoder?" Let me start by recalling what I know about encoder-decoder architectures. From the context provided, it's talking about layers in both the encoder and decoder. The encoder has N=6 layers, each with two sub-layers, and the decoder has the same number of layers but adds a third sub-layer for multi-head attention over the encoder's output.

First, I need to define what an encoder-decoder is in general. It's a framework used in neural networks, especially in tasks like machine translation or sequence-to-sequence problems. The encoder processes the input and creates a context vector, and the decoder uses that to generate the output.

Looking at the context, the encoder and decoder each have 6 layers. The encoder's layers have two sub-layers, probably self-attention and a feed-forward network, with residual connections and layer normalization. The decoder adds a third sub-layer that attends to the encoder's output. The model uses 